# Data Cleaning Pipeline

Cleans three data sources and writes canonical outputs to `cleaned/`:

| Source | Role | Output |
|---|---|---|
| SSUSA camera-trap records | primary | `ssusa_cleaned.csv` |
| IUCN terrestrial mammal ranges | primary | `iucn_cleaned.shp` |
| COMBINE traits ([Soria et al. 2021](https://doi.org/10.1002/ecy.3344)) | reference | none — merged in-memory |

Scope: loading, taxonomy reconciliation, synonym mapping, exclusion filtering,
body-mass enrichment, scope flags, per-source cleaned output, markdown summary.

All tunable parameters (paths, thresholds, species lists) live in **Cell 2**.


In [50]:
import gc
import logging
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

## Cell 2 — Parameters

Edit here to tune the pipeline.

In [51]:
# ---- File paths ----
SSUSA_SEQUENCES = "data/ssusa/ssusa_finalsequences.csv"
SSUSA_DEPLOYMENTS = "data/ssusa/ssusa_finaldeployments.csv"
IUCN_PATH = "data/MAMMALS_TERRESTRIAL_ONLY/MAMMALS_TERRESTRIAL_ONLY.shp"
TRAIT_PATH = "data/COMBINE/trait_data_imputed.csv"  # COMBINE (Soria et al. 2021)
OUTPUT_DIR = Path("cleaned")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---- IUCN filters ----
IUCN_PRESENCE_INCLUDE = [1, 2 , 3, 4, 6]       # 1=extant, 2=probably extant, 3=possibly extant, 4=possibly extinct, 6=presence uncertain
IUCN_ORIGIN_INCLUDE = [1, 2, 3, 5 ]   # 1=native, 2=reintroduced, 3=introduced, 5=origin uncertain
IUCN_SEASONAL_INCLUDE = [1, 2, 3, 4, 5]    # 1=resident, 2=breeding, 3=non-breeding seson, 4=migratory/passage , 5=seasonal occurrence uncertain
IUCN_BBOX_PADDING_DEG = 1.0       # pad around SSUSA extent before loading IUCN

# ---- Thresholds ----
BODY_MASS_THRESHOLD = 500         # grams

# ---- Synonyms (SSUSA/PanTHERIA name -> canonical normalized binomial) ----
SYNONYMS = {
    # Seeded from DataCleaning.ipynb IUCN remap:
    "mustela frenata":     "neogale frenata",      # long-tailed weasel
    "myodes gapperi":      "clethrionomys gapperi",# southern red-backed vole
    "neovison vison":      "neogale vison",        # american mink
    "pekania pennanti":    "martes pennanti",      # fisher
    # Bucket A (added from reconciliation review):
    "canis familiaris":        "canis lupus familiaris",  # domestic dog (older name)
    "capra aegagrus hircus":   "capra hircus",            # domestic goat (subspecies form)
    "cervus canadensis":       "cervus elaphus",          # elk (pre-2005 taxonomy)
    "sus scrofa scrofa":       "sus scrofa",              # wild boar (nominate subsp.)
}

# ---- Exclusion lists (normalized lowercase binomials) ----

SEMI_AQUATIC_MARINE = [
    "lontra canadensis",       # north american river otter
    "enhydra lutris",          # sea otter
    "castor canadensis",       # north american beaver
    "ondatra zibethicus",      # muskrat
    "neogale vison",           # american mink (post-synonym of neovison vison)
    "zalophus californianus",  # california sea lion
    "myocastor coypus",        # nutria / coypu
]

DOMESTIC_SPECIES = [
    "canis lupus familiaris",  # domestic dog
    "felis catus",             # domestic cat
    "bos taurus",              # cattle
    "equus caballus",          # horse
    "ovis aries",              # sheep
    "sus scrofa domesticus",   # domestic pig
    "capra hircus",            # domestic goat
    "equus asinus",            # donkey
    "rattus norvegicus",       # brown rat / norway rat
    "rattus rattus",           # house rat / black rat
    "mus musculus",            # house mouse
]

# introduced/escaped non-native wildlife (old pipeline dropped these).
NON_NATIVE_EXOTICS = [
    "boselaphus tragocamelus", # nilgai (introduced to Texas)
    "lama glama",              # llama (domestic/escaped, South American)
    "oryx gazella",            # gemsbok (introduced to New Mexico)
    "urva javanica",           # small asian mongoose (introduced to Hawaii)
    "lariscus insignis",       # three-striped ground squirrel (SE Asia — misID)
]

HUMANS = [
    "homo sapiens",            # human
]

# ---- Scope flags (NOT excluded — annotated in Scope_Flag column) ----
SCOPE_FLAGS = {
    "bison bison":  "managed_population",  # american bison (fenced herds dominate SSUSA detections)
}

## Cell 3 — Helper functions

In [52]:
def normalize_species(name):
    """Strip + lowercase + apply SYNONYMS dict."""
    if pd.isna(name):
        return name
    cleaned = str(name).strip().lower()
    return SYNONYMS.get(cleaned, cleaned)


def add_mass_flag(df, mass_col="body_mass_g", out_col="above_threshold"):
    """Add a bool column: mass >= BODY_MASS_THRESHOLD. Missing mass -> True
    (conservative: keep rather than silently drop when trait data is incomplete)."""

    df[out_col] = df[mass_col].isna() | (df[mass_col] >= BODY_MASS_THRESHOLD)
    return df


def add_scope_flags(df, spp_col="species_name", out_col="scope_flag"):
    """Map SCOPE_FLAGS dict onto a new column; empty string for unflagged rows."""
    df[out_col] = df[spp_col].map(SCOPE_FLAGS).fillna("")
    return df


# Union of all exclusion lists (used in Sections 3 and 4).
EXCLUDE_ALL = (
    set(SEMI_AQUATIC_MARINE)
    | set(DOMESTIC_SPECIES)
    | set(NON_NATIVE_EXOTICS)
    | set(HUMANS)
)
logger.info(
    "Exclusion set sizes: marine=%d, domestic=%d, exotic=%d, humans=%d, union=%d",
    len(SEMI_AQUATIC_MARINE), len(DOMESTIC_SPECIES),
    len(NON_NATIVE_EXOTICS), len(HUMANS), len(EXCLUDE_ALL),
)

INFO: Exclusion set sizes: marine=7, domestic=11, exotic=5, humans=1, union=24


## Section 1 — Load raw data

### 1a. SSUSA (sequences + deployments)

Nulls are encoded as a literal space `' '`, not NaN — pass `na_values=[' ']` or every dropna is a no-op.

In [53]:
# Tracks row counts at each stage for the final report.
waterfall = {}

seq = pd.read_csv(SSUSA_SEQUENCES, na_values=[" "], keep_default_na=True,
                  dtype={"Sequence_ID": "string"}, low_memory=False)
dep = pd.read_csv(SSUSA_DEPLOYMENTS, na_values=[" "], keep_default_na=True)
waterfall["sequences_raw"] = len(seq)
waterfall["deployments_raw"] = len(dep)
logger.info("SSUSA sequences raw: %s rows x %s cols", len(seq), seq.shape[1])
logger.info("SSUSA deployments raw: %s rows x %s cols", len(dep), dep.shape[1])

# Convert high-cardinality string columns to category to slash memory footprint
# (SSUSA repeats 'Mammalia', 'Carnivora', 'Snapshot USA 2019' etc. 850K+ times).
for _col in ["Project", "Camera_Trap_Array", "Class", "Order", "Family",
             "Genus", "Species", "Common_Name", "Age", "Sex"]:
    if _col in seq.columns:
        seq[_col] = seq[_col].astype("category")
for _col in ["Project", "Camera_Trap_Array", "Site_Name", "Habitat",
             "Development_Level", "Feature_Type"]:
    if _col in dep.columns:
        dep[_col] = dep[_col].astype("category")
gc.collect()

# Drop rows missing any taxonomy field we need to build Species_Name.
taxonomy_cols = ["Class", "Order", "Family", "Genus", "Species"]
seq = seq.dropna(subset=taxonomy_cols).reset_index(drop=True)
waterfall["after_dropna_taxonomy"] = len(seq)
logger.info("After dropna on %s: %s rows", taxonomy_cols, len(seq))

# Filter to mammals (case-insensitive).
seq = seq[seq["Class"].str.lower() == "mammalia"].reset_index(drop=True)
waterfall["after_mammalia_filter"] = len(seq)
logger.info("After Class=='mammalia': %s rows", len(seq))

# Build normalized binomial.
seq["Species_Name"] = (
    seq["Genus"].str.strip() + " " + seq["Species"].str.strip()
).str.lower()
seq["Species_Name"] = seq["Species_Name"].apply(normalize_species)

# Lowercase Age/Sex so "Unknown" vs "unknown" collapse; fill Group_Size.
seq["Age"] = seq["Age"].str.lower().fillna("unknown")
seq["Sex"] = seq["Sex"].str.lower().fillna("unknown")
seq["Group_Size"] = seq["Group_Size"].fillna(0)

# Inner-join sequences x deployments on the 4-key tuple.
merge_keys = ["Deployment_ID", "Year", "Project", "Camera_Trap_Array"]
before = len(seq)
ssusa = seq.merge(dep, on=merge_keys, how="inner")
waterfall["after_merge_deployments"] = len(ssusa)
logger.info("After inner-join with deployments (%s): %s -> %s rows",
            merge_keys, before, len(ssusa))

# Compute IUCN bbox from SSUSA footprint (includes AK + HI, not CONUS).
pad = IUCN_BBOX_PADDING_DEG
del seq, dep
gc.collect()

IUCN_BBOX = (
    float(ssusa["Longitude"].min()) - pad,
    float(ssusa["Latitude"].min())  - pad,
    float(ssusa["Longitude"].max()) + pad,
    float(ssusa["Latitude"].max())  + pad,
)
logger.info("Computed IUCN_BBOX from SSUSA extent (+%.1f deg pad): %s", pad, IUCN_BBOX)

print(f"SSUSA frame: {ssusa.shape}")
ssusa.head(3)

INFO: SSUSA sequences raw: 987979 rows x 16 cols
INFO: SSUSA deployments raw: 9679 rows x 13 cols
INFO: After dropna on ['Class', 'Order', 'Family', 'Genus', 'Species']: 891947 rows
INFO: After Class=='mammalia': 863436 rows
INFO: After inner-join with deployments (['Deployment_ID', 'Year', 'Project', 'Camera_Trap_Array']): 863436 -> 857003 rows
INFO: Computed IUCN_BBOX from SSUSA extent (+1.0 deg pad): (-158.74962, 20.355811, -67.61159314, 60.452635)


SSUSA frame: (857003, 26)


,Year,Project,Camera_Trap_Array,Deployment_ID,Sequence_ID,Start_Time,End_Time,Class,Order,Family,Genus,Species,Common_Name,Age,Sex,Group_Size,Species_Name,Site_Name,Start_Date,End_Date,Survey_Nights,Latitude,Longitude,Habitat,Development_Level,Feature_Type
0,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s1,2019/08/31 06:50:00,2019/08/31 06:50:00,Mammalia,Carnivora,Ursidae,Ursus,arctos,Brown Bear,unknown,unknown,1.0,ursus arctos,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64,59.42643,-136.2225,Forest,Wild,Water source
1,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s2,2019/08/31 14:15:00,2019/08/31 14:17:00,Mammalia,Carnivora,Ursidae,Ursus,arctos,Brown Bear,unknown,unknown,1.0,ursus arctos,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64,59.42643,-136.2225,Forest,Wild,Water source
2,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s3,2019/08/31 18:22:00,2019/08/31 18:22:00,Mammalia,Carnivora,Ursidae,Ursus,arctos,Brown Bear,unknown,unknown,1.0,ursus arctos,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64,59.42643,-136.2225,Forest,Wild,Water source


### 1b. IUCN range polygons

Filtered by SSUSA-derived bbox, then by presence/origin/seasonal.

In [60]:
iucn = gpd.read_file(IUCN_PATH, bbox=IUCN_BBOX)
waterfall["iucn_after_bbox_load"] = len(iucn)
waterfall["iucn_species_after_bbox_load"] = iucn["sci_name"].nunique()
logger.info("IUCN after bbox load: %s polygons, %s species",
            len(iucn), iucn["sci_name"].nunique())
print("IUCN columns:", list(iucn.columns))

# Normalize species in place (keep column name 'sci_name' for DBF <=10 char fit).
iucn["sci_name"] = iucn["sci_name"].apply(normalize_species)

# Rename order_ -> order for consistency with PanTHERIA / SSUSA Order.
iucn = iucn.rename(columns={"order_": "order"})

# Apply presence/origin/seasonal filters.
before = len(iucn)
iucn = iucn[
    iucn["presence"].isin(IUCN_PRESENCE_INCLUDE)
    & iucn["origin"].isin(IUCN_ORIGIN_INCLUDE)
    & iucn["seasonal"].isin(IUCN_SEASONAL_INCLUDE)
].reset_index(drop=True)
waterfall["iucn_after_filter"] = len(iucn)
waterfall["iucn_species_after_filter"] = iucn["sci_name"].nunique()
logger.info("After presence/origin/seasonal filter: %s -> %s polygons (%s species)",
            before, len(iucn), iucn["sci_name"].nunique())

# Retain wide attribute set (drop only recomputable SHAPE_* and constant kingdom/phylum).
keep_cols = [
    "id_no", "sci_name", "presence", "origin", "seasonal", "category",
    "compiler", "yrcompiled", "citation", "subspecies", "subpop", "source",
    "island", "tax_comm", "dist_comm", "generalisd", "legend",
    "class", "order", "family", "genus",
    "marine", "terrestria", "freshwater",
    "geometry",
]
iucn = iucn[[c for c in keep_cols if c in iucn.columns]]
print(f"IUCN frame: {iucn.shape}")
iucn.drop(columns="geometry").head(3)

INFO: IUCN after bbox load: 767 polygons, 580 species
INFO: After presence/origin/seasonal filter: 767 -> 753 polygons (580 species)


IUCN columns: ['id_no', 'sci_name', 'presence', 'origin', 'seasonal', 'compiler', 'yrcompiled', 'citation', 'subspecies', 'subpop', 'source', 'island', 'tax_comm', 'dist_comm', 'generalisd', 'legend', 'kingdom', 'phylum', 'class', 'order_', 'family', 'genus', 'category', 'marine', 'terrestria', 'freshwater', 'SHAPE_Leng', 'SHAPE_Area', 'geometry']
IUCN frame: (753, 25)


,id_no,sci_name,presence,origin,seasonal,category,compiler,yrcompiled,citation,subspecies,subpop,source,island,tax_comm,dist_comm,generalisd,legend,class,order,family,genus,marine,terrestria,freshwater
0,699,cuniculus paca,1,1,1,LC,IUCN SSC Small Mammal Specialist Group,2016,IUCN SSC Small Mammal Specialist Group,None,None,None,None,None,None,0,Extant (resident),MAMMALIA,RODENTIA,CUNICULIDAE,Cuniculus,false,true,false
1,699,cuniculus paca,1,3,1,LC,IUCN SSC Small Mammal Specialist Group,2016,IUCN SSC Small Mammal Specialist Group,None,None,None,Cuba,None,None,0,Extant & Introduced (resident),MAMMALIA,RODENTIA,CUNICULIDAE,Cuniculus,false,true,false
2,914,alouatta pigra,1,1,1,EN,IUCN,2020,IUCN (International Union for Conservation of Nature),None,None,None,None,None,None,0,Extant (resident),MAMMALIA,PRIMATES,ATELIDAE,Alouatta,false,true,false


### 1c. COMBINE traits (reference only, not written back)

[COMBINE](https://doi.org/10.6084/m9.figshare.13028255) (Soria et al. 2021, *Ecology*) — 6,263 terrestrial mammal species with imputed functional traits.

In [61]:
trait_raw = pd.read_csv(TRAIT_PATH)
logger.info("COMBINE raw: %s rows x %s cols", *trait_raw.shape)

wanted = {
    "order":                    "order",
    "family":                   "family",
    "iucn2020_binomial":        "species_name",
    "adult_mass_g":             "body_mass_g",
    "adult_body_length_mm":     "head_body_len_mm",
}
pan = trait_raw[list(wanted.keys())].rename(columns=wanted).copy()

# COMBINE uses NA (not a sentinel). Imputed file has ~100% body-mass coverage.
# Normalize species for joinability.
pan["species_name"] = pan["species_name"].apply(normalize_species)

# Drop rows without a species_name (e.g. taxonomy placeholders).
pan = pan[pan["species_name"].notna() & (pan["species_name"] != "")].copy()

# Deduplicate: after normalize_species + synonym collapse, some rows may now
# share the same canonical binomial. Keep the first (empirical first in COMBINE).
before = len(pan)
pan = pan.drop_duplicates(subset="species_name", keep="first").reset_index(drop=True)
if before != len(pan):
    logger.info("Dropped %s duplicate trait rows after synonym normalization",
                before - len(pan))

missing_mass = pan["body_mass_g"].isna().sum()
logger.info("COMBINE usable: %s species, %s missing body mass (%.1f%%)",
            len(pan), missing_mass, 100 * missing_mass / max(len(pan), 1))

pan.head(3)

INFO: COMBINE raw: 6263 rows x 60 cols
INFO: Dropped 302 duplicate trait rows after synonym normalization
INFO: COMBINE usable: 5961 species, 217 missing body mass (3.6%)


,order,family,species_name,body_mass_g,head_body_len_mm
0,Rodentia,Muridae,abditomys latidens,268.09,228.000
1,Rodentia,Muridae,abeomelomys sevia,57.89,158.645
2,Rodentia,Cricetidae,abrawayaomys chebezi,NaN,NaN


## Section 2 — Taxonomy reconciliation

Cross-reference the three sources by normalized binomial. No CSV written —
the DataFrame is displayed inline, and summary counts flow into the final
markdown report.

In [62]:
ssusa_spp = set(ssusa["Species_Name"].dropna().unique())
iucn_spp = set(iucn["sci_name"].dropna().unique())
pan_spp = set(pan["species_name"].dropna().unique())

all_spp = sorted(ssusa_spp | iucn_spp | pan_spp)
recon = pd.DataFrame({"species_name": all_spp})
recon["in_ssusa"] = recon["species_name"].isin(ssusa_spp)
recon["in_iucn"] = recon["species_name"].isin(iucn_spp)
recon["in_combine"] = recon["species_name"].isin(pan_spp)

# SSUSA record counts per species.
ssusa_counts = ssusa["Species_Name"].value_counts().to_dict()
recon["ssusa_records"] = recon["species_name"].map(ssusa_counts).fillna(0).astype(int)

# PanTHERIA body mass.
pan_mass = dict(zip(pan["species_name"], pan["body_mass_g"]))
recon["body_mass_g"] = recon["species_name"].map(pan_mass)

three_way = (recon["in_ssusa"] & recon["in_iucn"] & recon["in_combine"]).sum()
ssusa_only = recon[recon["in_ssusa"] & ~recon["in_iucn"]]
missing_mass = recon[recon["in_ssusa"] & ~recon["body_mass_g"].notna()]

logger.info("Reconciliation: %s total species across sources", len(recon))
logger.info("  all three sources: %s", three_way)
logger.info("  in SSUSA but not IUCN: %s", len(ssusa_only))
logger.info("  in SSUSA but missing body mass: %s", len(missing_mass))

recon_summary = {
    "total_species": len(recon),
    "three_way": int(three_way),
    "ssusa_only_vs_iucn": ssusa_only["species_name"].tolist(),
    "ssusa_missing_mass": missing_mass["species_name"].tolist(),
    "synonyms_applied": dict(SYNONYMS),
}

recon

INFO: Reconciliation: 5967 total species across sources
INFO:   all three sources: 108
INFO:   in SSUSA but not IUCN: 1
INFO:   in SSUSA but missing body mass: 0


,species_name,in_ssusa,in_iucn,in_combine,ssusa_records,body_mass_g
0,abditomys latidens,False,False,True,0,268.090
1,abeomelomys sevia,False,False,True,0,57.890
2,abrawayaomys chebezi,False,False,True,0,NaN
3,abrawayaomys ruschii,False,False,True,0,62.990
4,abrocoma bennettii,False,False,True,0,265.500
...,...,...,...,...,...,...
5962,zyzomys argurus,False,False,True,0,45.210
5963,zyzomys maini,False,False,True,0,93.995
5964,zyzomys palatalis,False,False,True,0,123.000
5965,zyzomys pedunculatus,False,False,True,0,100.000


### SSUSA species not in IUCN (manual review candidates)

Each row below is either a synonym that needs adding to `SYNONYMS`, or a genuine snapshot-only record (e.g. escaped exotic, misidentification).

In [63]:
ssusa_only.sort_values("ssusa_records", ascending=False).head(30)

,species_name,in_ssusa,in_iucn,in_combine,ssusa_records,body_mass_g
5496,sus scrofa,True,False,True,15942,135000.0


## Section 3 — Clean SSUSA and write `ssusa_cleaned.csv`

In [64]:
# Dedup exact-duplicate rows.
before = len(ssusa)
ssusa = ssusa.drop_duplicates().reset_index(drop=True)
waterfall["after_dedup"] = len(ssusa)
logger.info("Dedup: %s -> %s rows (%s removed)", before, len(ssusa), before - len(ssusa))

# Exclusion tallies per list (for the final report).
exclusion_counts = {
    "SEMI_AQUATIC_MARINE": int(ssusa["Species_Name"].isin(set(SEMI_AQUATIC_MARINE)).sum()),
    "DOMESTIC_SPECIES":    int(ssusa["Species_Name"].isin(set(DOMESTIC_SPECIES)).sum()),
    "NON_NATIVE_EXOTICS":  int(ssusa["Species_Name"].isin(set(NON_NATIVE_EXOTICS)).sum()),
    "HUMANS":              int(ssusa["Species_Name"].isin(set(HUMANS)).sum()),
}
for k, v in exclusion_counts.items():
    logger.info("SSUSA to exclude via %s: %s records", k, v)

before = len(ssusa)
ssusa = ssusa[~ssusa["Species_Name"].isin(EXCLUDE_ALL)].reset_index(drop=True)
waterfall["after_exclusions"] = len(ssusa)
logger.info("After union exclusion: %s -> %s rows", before, len(ssusa))

# Attach body mass from COMBINE on normalized binomial.
# Drop any previous mass column first so rerunning this cell is idempotent.
mass_lookup = pan.set_index("species_name")["body_mass_g"]
ssusa = ssusa.drop(columns=["Body_Mass_g", "body_mass_g"], errors="ignore")
ssusa["Body_Mass_g"] = ssusa["Species_Name"].map(mass_lookup)
n_missing = ssusa["Body_Mass_g"].isna().sum()
logger.info("Merged COMBINE body mass; %s records have no mass (%.1f%%)",
            n_missing, 100 * n_missing / len(ssusa))

# Flags.
ssusa = add_mass_flag(ssusa, mass_col="Body_Mass_g", out_col="Above_Threshold")
ssusa = add_scope_flags(ssusa, spp_col="Species_Name", out_col="Scope_Flag")

# Order final columns: natives first (Pascal_Case), then derived.
final_cols = [
    "Year", "Project", "Camera_Trap_Array", "Deployment_ID", "Sequence_ID",
    "Start_Time", "End_Time", "Class", "Order", "Family", "Genus", "Species",
    "Common_Name", "Age", "Sex", "Group_Size",
    "Site_Name", "Start_Date", "End_Date", "Survey_Nights",
    "Latitude", "Longitude", "Habitat", "Development_Level", "Feature_Type",
    "Species_Name", "Body_Mass_g", "Above_Threshold", "Scope_Flag",
]
ssusa = ssusa[[c for c in final_cols if c in ssusa.columns]]

ssusa_out = OUTPUT_DIR / "ssusa_cleaned.csv"
ssusa.to_csv(ssusa_out, index=False)
logger.info("Wrote %s (%s rows, %s cols)", ssusa_out, len(ssusa), ssusa.shape[1])

ssusa.head(3)

# snapshot output frame shape *before* the memory trim so the
# report can state the on-disk column count (not the trimmed one).
ssusa_out_rows, ssusa_out_cols = ssusa.shape
ssusa_out_species = ssusa['Species_Name'].nunique()

# release memory before Section 4 writes the big IUCN shapefile
ssusa_slim = ssusa[["Species_Name", "Body_Mass_g", "Above_Threshold", "Scope_Flag"]].copy()
del ssusa
ssusa = ssusa_slim  # only the columns Section 6 report needs
del ssusa_slim
gc.collect()

ssusa.head(3)

INFO: Dedup: 713319 -> 109 rows (713210 removed)
INFO: SSUSA to exclude via SEMI_AQUATIC_MARINE: 0 records
INFO: SSUSA to exclude via DOMESTIC_SPECIES: 0 records
INFO: SSUSA to exclude via NON_NATIVE_EXOTICS: 0 records
INFO: SSUSA to exclude via HUMANS: 0 records
INFO: After union exclusion: 109 -> 109 rows
INFO: Merged COMBINE body mass; 0 records have no mass (0.0%)
INFO: Wrote cleaned/ssusa_cleaned.csv (109 rows, 4 cols)


,Species_Name,Body_Mass_g,Above_Threshold,Scope_Flag
0,ursus arctos,240500.0,True,
1,alces alces,359750.0,True,
2,canis latrans,11050.0,True,


## Section 4 — Clean IUCN and write `iucn_cleaned.shp`

In [ ]:
iucn_excl_counts = {
    "SEMI_AQUATIC_MARINE": int(iucn["sci_name"].isin(set(SEMI_AQUATIC_MARINE)).sum()),
    "DOMESTIC_SPECIES":    int(iucn["sci_name"].isin(set(DOMESTIC_SPECIES)).sum()),
    "NON_NATIVE_EXOTICS":  int(iucn["sci_name"].isin(set(NON_NATIVE_EXOTICS)).sum()),
    "HUMANS":              int(iucn["sci_name"].isin(set(HUMANS)).sum()),
}
for k, v in iucn_excl_counts.items():
    logger.info("IUCN to exclude via %s: %s polygons", k, v)

before = len(iucn)
iucn = iucn[~iucn["sci_name"].isin(EXCLUDE_ALL)].reset_index(drop=True)
logger.info("IUCN after union exclusion: %s -> %s polygons", before, len(iucn))

# Attach body mass (lowercase on the IUCN side).
# Drop any previous mass columns first so rerunning this cell cannot create duplicate labels.
mass_lookup = pan.set_index("species_name")["body_mass_g"]
iucn = iucn.drop(columns=["body_mass", "body_mass_g", "species_name"], errors="ignore")
iucn["body_mass"] = iucn["sci_name"].map(mass_lookup)

# Check how many species are missing mass after the merge.
n_missing = iucn["body_mass"].isna().sum()
logger.info("Merged COMBINE body mass into IUCN; %s polygons have no mass (%.1f%%)",
            n_missing, 100 * n_missing / len(iucn))

# print species names missing mass in logger
missing_mass_iucn = iucn[iucn["body_mass"].isna()]["sci_name"].unique()
logger.info("IUCN species missing mass after merge: %s", missing_mass_iucn.tolist())    

# remove species with missing masses as their body masses are less than 500 g 
iucn = iucn[~iucn["sci_name"].isin(missing_mass_iucn)].reset_index(drop=True)
logger.info("After removing species with missing mass: %s polygons",
            len(iucn))

# Flags.
# add in logger iucn add mass flag and scope flag counts
logger.info("Adding mass flag (Body_Mass >= %s g) and scope flags from SCOPE_FLAGS dict",
            BODY_MASS_THRESHOLD)
iucn = add_mass_flag(iucn, mass_col="body_mass", out_col="above_threshold")
iucn = iucn.rename(columns={"above_threshold": "ab_thres"})
iucn = add_scope_flags(iucn, spp_col="sci_name", out_col="scope_flag")  # scope_flag is exactly 10 chars

# Verify every column name fits the DBF 10-char cap.
too_long = [c for c in iucn.columns if c != "geometry" and len(c) > 10]
if too_long:
    raise ValueError(f"Column names exceed DBF 10-char limit: {too_long}")

iucn_out = OUTPUT_DIR / "iucn_cleaned.shp"
iucn.to_file(iucn_out)
logger.info("Wrote %s (%s polygons, %s species)",
            iucn_out, len(iucn), iucn["sci_name"].nunique())

iucn.drop(columns="geometry").head(3)

INFO: IUCN to exclude via SEMI_AQUATIC_MARINE: 0 polygons
INFO: IUCN to exclude via DOMESTIC_SPECIES: 2 polygons
INFO: IUCN to exclude via NON_NATIVE_EXOTICS: 0 polygons
INFO: IUCN to exclude via HUMANS: 0 polygons
INFO: IUCN after union exclusion: 753 -> 751 polygons
INFO: Merged COMBINE body mass into IUCN; 7 polygons have no mass (0.9%)
INFO: IUCN species missing mass after merge: ['clethrionomys rutilus', 'diaemus youngii', 'alexandromys oeconomus', 'sorex monticola', 'clethrionomys californicus', 'molossus alvarezi', 'cyclopes dorsalis']
INFO: After removing species with missing mass: 744 polygons, 571 species
INFO: Adding mass flag (Body_Mass >= 500 g) and scope flags from SCOPE_FLAGS dict
INFO: Created 744 records
INFO: Wrote cleaned/iucn_cleaned.shp (744 polygons, 571 species)


,id_no,sci_name,presence,origin,seasonal,category,compiler,yrcompiled,citation,subspecies,subpop,source,island,tax_comm,dist_comm,generalisd,legend,class,order,family,genus,marine,terrestria,freshwater,body_mass,ab_thres,scope_flag
0,699,cuniculus paca,1,1,1,LC,IUCN SSC Small Mammal Specialist Group,2016,IUCN SSC Small Mammal Specialist Group,None,None,None,None,None,None,0,Extant (resident),MAMMALIA,RODENTIA,CUNICULIDAE,Cuniculus,false,true,false,9000.00,True,
1,699,cuniculus paca,1,3,1,LC,IUCN SSC Small Mammal Specialist Group,2016,IUCN SSC Small Mammal Specialist Group,None,None,None,Cuba,None,None,0,Extant & Introduced (resident),MAMMALIA,RODENTIA,CUNICULIDAE,Cuniculus,false,true,false,9000.00,True,
2,914,alouatta pigra,1,1,1,EN,IUCN,2020,IUCN (International Union for Conservation of Nature),None,None,None,None,None,None,0,Extant (resident),MAMMALIA,PRIMATES,ATELIDAE,Alouatta,false,true,false,9386.03,True,


## Section 5 — Final summary (`data_cleaning_report.md`)

Auto-generated markdown, rendered inline at the bottom.

In [66]:
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def _md_list(items, empty="(none)"):
    items = list(items)
    if not items:
        return empty
    return "".join(f"- {x}\n" for x in items)

threshold_total = ssusa_out_rows
threshold_above = int(ssusa["Above_Threshold"].sum())  # after trim, still present
threshold_below = threshold_total - threshold_above

report = f"""# Data Cleaning Report

Generated: {timestamp}

## Raw data
- SSUSA sequences: {waterfall['sequences_raw']:,} rows
- SSUSA deployments: {waterfall['deployments_raw']:,} rows
- IUCN polygons (after bbox load): {waterfall["iucn_after_bbox_load"]:,} ({waterfall["iucn_species_after_bbox_load"]} species)
- IUCN polygons (after filter + exclusion): {len(iucn):,} ({iucn["sci_name"].nunique()} species)
- COMBINE ([Soria et al. 2021](https://doi.org/10.1002/ecy.3344)): {len(pan):,} species, {pan['body_mass_g'].notna().sum():,} with body mass

## SSUSA cleaning waterfall
| Stage | Rows |
|---|---|
| raw sequences | {waterfall['sequences_raw']:,} |
| after dropna on taxonomy | {waterfall['after_dropna_taxonomy']:,} |
| after Class == mammalia | {waterfall['after_mammalia_filter']:,} |
| after merge with deployments | {waterfall['after_merge_deployments']:,} |
| after dedup | {waterfall['after_dedup']:,} |
| after exclusion lists | {waterfall['after_exclusions']:,} |

## IUCN bbox (computed from SSUSA footprint)
```
minx, miny, maxx, maxy = {IUCN_BBOX}
padding = {IUCN_BBOX_PADDING_DEG} degrees
```

## Taxonomy reconciliation
- Total unique species across all three sources: **{recon_summary['total_species']}**
- Present in all three sources: **{recon_summary['three_way']}**
- In SSUSA but not IUCN ({len(recon_summary['ssusa_only_vs_iucn'])} species):

{_md_list(recon_summary['ssusa_only_vs_iucn'])}
- In SSUSA but missing body mass ({len(recon_summary['ssusa_missing_mass'])} species):

{_md_list(recon_summary['ssusa_missing_mass'])}
- Synonyms applied ({len(recon_summary['synonyms_applied'])}):

{_md_list(f"`{k}` -> `{v}`" for k, v in recon_summary['synonyms_applied'].items())}

## Exclusion tallies
### SSUSA records dropped
| List | Records |
|---|---|
| SEMI_AQUATIC_MARINE | {exclusion_counts['SEMI_AQUATIC_MARINE']:,} |
| DOMESTIC_SPECIES | {exclusion_counts['DOMESTIC_SPECIES']:,} |
| NON_NATIVE_EXOTICS | {exclusion_counts['NON_NATIVE_EXOTICS']:,} |
| HUMANS | {exclusion_counts['HUMANS']:,} |

### IUCN polygons dropped
| List | Polygons |
|---|---|
| SEMI_AQUATIC_MARINE | {iucn_excl_counts['SEMI_AQUATIC_MARINE']:,} |
| DOMESTIC_SPECIES | {iucn_excl_counts['DOMESTIC_SPECIES']:,} |
| NON_NATIVE_EXOTICS | {iucn_excl_counts['NON_NATIVE_EXOTICS']:,} |
| HUMANS | {iucn_excl_counts['HUMANS']:,} |

## Scope-flagged species (retained, not excluded)
{_md_list(f"`{k}` -> {v}" for k, v in SCOPE_FLAGS.items())}

## Cleaned outputs
- `ssusa_cleaned.csv`: **{ssusa_out_rows:,} rows**, {ssusa_out_cols} columns, {ssusa_out_species} species
- `iucn_cleaned.shp`: **{len(iucn):,} polygons**, {iucn.shape[1] - 1} attribute columns, {iucn['sci_name'].nunique()} species

## Body-mass threshold summary
- Threshold: **{BODY_MASS_THRESHOLD} g**
- SSUSA records: {threshold_total:,}
- Records above threshold: {threshold_above:,}
- Records below threshold: {threshold_below:,}
"""

report_path = OUTPUT_DIR / "data_cleaning_report.md"
report_path.write_text(report)
logger.info("Wrote %s", report_path)
# print(report)

INFO: Wrote cleaned/data_cleaning_report.md
